# UK Yield Curve Recession Forecasting

## Data collection and initial inspection

This notebook loads and inspects the UK yield-curve and real-GDP data used in the analysis.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Imports successful")

Imports successful


In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder exists: {DATA_RAW.exists()}")
print(f"Processed data folder exists: {DATA_PROCESSED.exists()}")
print(f"Figures folder exists: {OUTPUT_FIGURES.exists()}")

Project root: /Users/lohith/Documents/GitHub/UK-Yield-Curve-Recession-Forecasting
Raw data folder exists: True
Processed data folder exists: True
Figures folder exists: True


## Bank of England yield-curve data

The Bank of England archive stores nominal daily yield-curve estimates across eight Excel workbooks. Before loading the observations, we inspect the available files and worksheet names.

In [4]:
BOE_DIR = DATA_RAW / "boe" / "glcnominalddata"

boe_files = sorted(BOE_DIR.glob("*.xlsx"))

print(f"Number of workbooks found: {len(boe_files)}")

for file_path in boe_files:
    print(file_path.name)

Number of workbooks found: 8
GLC Nominal daily data_1979 to 1984.xlsx
GLC Nominal daily data_1985 to 1989.xlsx
GLC Nominal daily data_1990 to 1994.xlsx
GLC Nominal daily data_1995 to 1999.xlsx
GLC Nominal daily data_2000 to 2004.xlsx
GLC Nominal daily data_2005 to 2015.xlsx
GLC Nominal daily data_2016 to 2024.xlsx
GLC Nominal daily data_2025 to present.xlsx


In [5]:
for file_path in boe_files:
    workbook = pd.ExcelFile(file_path)

    print(f"\n{file_path.name}")

    for sheet_name in workbook.sheet_names:
        print(f"  - {sheet_name}")


GLC Nominal daily data_1979 to 1984.xlsx
  - info
  - 1. nominal fwds, short end
  - 2. nominal fwd curve
  - 3. nominal spot, short end
  - 4. nominal spot curve

GLC Nominal daily data_1985 to 1989.xlsx
  - info
  - 1. nominal fwds, short end
  - 2. nominal fwd curve
  - 3. nominal spot, short end
  - 4. nominal spot curve

GLC Nominal daily data_1990 to 1994.xlsx
  - info
  - 1. nominal fwds, short end
  - 2. nominal fwd curve
  - 3. nominal spot, short end
  - 4. nominal spot curve

GLC Nominal daily data_1995 to 1999.xlsx
  - info
  - 1. nominal fwds, short end
  - 2. nominal fwd curve
  - 3. nominal spot, short end
  - 4. nominal spot curve

GLC Nominal daily data_2000 to 2004.xlsx
  - info
  - 1. nominal fwds, short end
  - 2. nominal fwd curve
  - 3. nominal spot, short end
  - 4. nominal spot curve

GLC Nominal daily data_2005 to 2015.xlsx
  - info
  - 1. fwds, short end
  - 2. fwd curve
  - 3. spot, short end
  - 4. spot curve

GLC Nominal daily data_2016 to 2024.xlsx
  - in

In [6]:
def find_spot_curve_sheet(file_path):
    workbook = pd.ExcelFile(file_path)

    matching_sheets = [
        sheet
        for sheet in workbook.sheet_names
        if sheet.endswith("spot curve")
    ]

    if len(matching_sheets) != 1:
        raise ValueError(
            f"Expected one spot-curve sheet in {file_path.name}, "
            f"but found {matching_sheets}"
        )

    return matching_sheets[0]

In [7]:
files_to_preview = [
    boe_files[0],
    boe_files[-1],
]

for file_path in files_to_preview:
    sheet_name = find_spot_curve_sheet(file_path)

    preview = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        nrows=10,
    )

    print(f"\nWorkbook: {file_path.name}")
    print(f"Worksheet: {sheet_name}")

    display(preview.iloc[:, :25])


Workbook: GLC Nominal daily data_1979 to 1984.xlsx
Worksheet: 4. nominal spot curve


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,UK nominal spot curve,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Maturity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,years:,0.5,1.000000,1.500000,2.000000,2.500000,3.000000,3.500000,4.000000,4.500000,...,7.500000,8.000000,8.500000,9.000000,9.500000,10.000000,10.500000,11.000000,11.500000,12.000000
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1979-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1979-01-02 00:00:00,NaN,11.446775,11.733149,11.959974,12.133409,12.266092,12.368615,12.448401,12.510498,...,12.657346,12.657866,12.654011,12.646479,12.635924,12.622968,12.608204,12.592209,12.575541,12.558744
7,1979-01-03 00:00:00,NaN,11.521413,11.770976,11.987385,12.160734,12.296541,12.402607,12.485395,12.549727,...,12.698820,12.698748,12.694202,12.685917,12.674578,12.660829,12.645285,12.628535,12.611154,12.593692
8,1979-01-04 00:00:00,NaN,11.517242,11.739487,11.941209,12.108479,12.243433,12.351444,12.437431,12.505331,...,12.669599,12.670754,12.667166,12.659623,12.648856,12.635555,12.620382,12.603973,12.586946,12.569891
9,1979-01-05 00:00:00,NaN,11.463236,11.683237,11.885243,12.055542,12.195494,12.309677,12.402443,12.477333,...,12.678973,12.685042,12.685940,12.682451,12.675311,12.665215,12.652834,12.638814,12.623785,12.608353



Workbook: GLC Nominal daily data_2025 to present.xlsx
Worksheet: 4. spot curve


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,UK nominal spot curve,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Maturity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,years:,0.5,1,1.5,2,2.5,3,3.5,4,4.5,...,7.5,8,8.5,9,9.5,10,10.5,11,11.5,12
4,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,...,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh,Refresh
5,2025-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-01-02 00:00:00,4.458443,4.309412,4.229347,4.198192,4.182999,4.176881,4.178554,4.187243,4.202204,...,4.381008,4.41927,4.458436,4.498105,4.537952,4.577716,4.617184,4.65617,4.694513,4.732064
7,2025-01-03 00:00:00,4.466675,4.319469,4.246626,4.221993,4.210895,4.206675,4.208586,4.216353,4.229652,...,4.396148,4.432937,4.470858,4.509491,4.548488,4.587559,4.626465,4.664993,4.702958,4.740189
8,2025-01-06 00:00:00,4.459946,4.322789,4.259158,4.240341,4.232715,4.230632,4.233921,4.242621,4.256586,...,4.424541,4.461314,4.499178,4.537724,4.576617,4.615576,4.654371,4.692801,4.730688,4.76787
9,2025-01-07 00:00:00,4.463509,4.328843,4.27211,4.260181,4.25859,4.261658,4.269351,4.281856,4.299142,...,4.47976,4.517665,4.556437,4.595694,4.635123,4.674471,4.71353,4.752121,4.790087,4.827283


In [11]:
def load_boe_workbook(file_path):
    sheet_name = find_spot_curve_sheet(file_path)

    raw = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=3,
    )

    raw = raw.rename(columns={raw.columns[0]: "date"})

    required_columns = ["date", 2.0, 10.0]
    missing_columns = [
        column for column in required_columns
        if column not in raw.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: {missing_columns}"
        )

    selected = raw[required_columns].copy()

    selected.columns = [
        "date",
        "yield_2y",
        "yield_10y",
    ]

    selected["date"] = pd.to_datetime(
        selected["date"],
        format = "mixed",
        errors="coerce",
    )

    selected["yield_2y"] = pd.to_numeric(
        selected["yield_2y"],
        errors="coerce",
    )

    selected["yield_10y"] = pd.to_numeric(
        selected["yield_10y"],
        errors="coerce",
    )

    selected = selected.dropna(subset=["date"])
    selected["source_file"] = file_path.name

    return selected

In [12]:
boe_daily = pd.concat(
    [load_boe_workbook(file_path) for file_path in boe_files],
    ignore_index=True,
)

boe_daily = boe_daily.sort_values("date").reset_index(drop=True)

print(f"Total dated rows: {len(boe_daily):,}")
print(f"First date: {boe_daily['date'].min().date()}")
print(f"Last date: {boe_daily['date'].max().date()}")
print(f"Duplicate dates: {boe_daily['date'].duplicated().sum():,}")

print("\nMissing values:")
print(boe_daily[["yield_2y", "yield_10y"]].isna().sum())

Total dated rows: 12,435
First date: 1979-01-01
Last date: 2026-08-28
Duplicate dates: 0

Missing values:
yield_2y     389
yield_10y    389
dtype: int64


In [10]:
for column in ["yield_2y", "yield_10y"]:
    available = boe_daily.loc[
        boe_daily[column].notna(),
        ["date", column],
    ]

    print(f"\n{column}")
    print(f"  First available date: {available['date'].min().date()}")
    print(f"  Last available date:  {available['date'].max().date()}")
    print(f"  Available observations: {len(available):,}")

jointly_available = boe_daily.dropna(
    subset=["yield_2y", "yield_10y"]
)

print("\nJointly available observations")
print(f"  First date: {jointly_available['date'].min().date()}")
print(f"  Last date:  {jointly_available['date'].max().date()}")
print(f"  Observations: {len(jointly_available):,}")


yield_2y
  First available date: 1979-01-02
  Last available date:  2026-08-28
  Available observations: 12,046

yield_10y
  First available date: 1979-01-02
  Last available date:  2026-08-28
  Available observations: 12,046

Jointly available observations
  First date: 1979-01-02
  Last date:  2026-08-28
  Observations: 12,046


## Construct quarterly yield spreads

Daily observations with missing yields are removed before calculating quarterly average 2-year and 10-year spot yields. The principal yield spread is calculated as the quarterly average 10-year yield minus the quarterly average 2-year yield.

In [14]:
boe_clean = boe_daily.dropna(
    subset=["yield_2y", "yield_10y"]
).copy()

boe_clean["quarter"] = boe_clean["date"].dt.to_period("Q")

boe_quarterly = (
    boe_clean
    .groupby("quarter", as_index=False)
    .agg(
        yield_2y=("yield_2y", "mean"),
        yield_10y=("yield_10y", "mean"),
        trading_days=("date", "count"),
    )
)

boe_quarterly["spread_10y_2y"] = (
    boe_quarterly["yield_10y"]
    - boe_quarterly["yield_2y"]
)

boe_quarterly = boe_quarterly[
    [
        "quarter",
        "yield_2y",
        "yield_10y",
        "spread_10y_2y",
        "trading_days",
    ]
]

In [15]:
print(f"Number of quarters: {len(boe_quarterly)}")
print(f"First quarter: {boe_quarterly['quarter'].min()}")
print(f"Last quarter: {boe_quarterly['quarter'].max()}")

print("\nMissing values:")
print(boe_quarterly.isna().sum())

print("\nTrading days per quarter:")
print(boe_quarterly["trading_days"].describe())

display(boe_quarterly.head())
display(boe_quarterly.tail())

Number of quarters: 191
First quarter: 1979Q1
Last quarter: 2026Q3

Missing values:
quarter          0
yield_2y         0
yield_10y        0
spread_10y_2y    0
trading_days     0
dtype: int64

Trading days per quarter:
count    191.000000
mean      63.068063
std        2.034072
min       43.000000
25%       62.000000
50%       64.000000
75%       64.000000
max       65.000000
Name: trading_days, dtype: float64


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days
0,1979Q1,11.827995,12.679240,0.851246,64
1,1979Q2,10.803278,11.498352,0.695075,61
2,1979Q3,11.717763,11.913749,0.195985,64
3,1979Q4,13.503208,13.197965,-0.305243,64
4,1980Q1,14.502812,13.655117,-0.847694,64


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days
186,2025Q3,3.755126,4.679847,0.924721,65
187,2025Q4,3.680211,4.584931,0.904720,64
188,2026Q1,3.725403,4.623884,0.898481,63
189,2026Q2,4.193059,4.935797,0.742738,61
190,2026Q3,4.221020,5.038128,0.817108,43


In [16]:
boe_output_path = DATA_PROCESSED / "boe_quarterly_yields.csv"

boe_quarterly.to_csv(
    boe_output_path,
    index=False,
)

print(f"Saved to: {boe_output_path}")

Saved to: /Users/lohith/Documents/GitHub/UK-Yield-Curve-Recession-Forecasting/data/processed/boe_quarterly_yields.csv
